In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
!pip install transformers[torch] datasets evaluate rouge_score peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [4]:
import torch
from datasets import load_dataset
from transformers import (
    BartForConditionalGeneration, 
    BartTokenizer, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
from peft import LoraConfig, get_peft_model, TaskType

In [5]:
from huggingface_hub import notebook_login

# This will create a popup box for you to paste your token
notebook_login()

In [6]:
model_ckpt = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_ckpt)
model = BartForConditionalGeneration.from_pretrained(model_ckpt)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [7]:
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 884,736 || all params: 140,305,152 || trainable%: 0.6306


In [8]:
dataset = load_dataset("cnn_dailymail", "3.0.0")
train_data = dataset["train"].shuffle(seed=42).select(range(59000))
val_data = dataset["validation"].select(range(2000)) # Smaller val set for speed

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [9]:
def preprocess_function(examples):
    # 1. Tokenize the input articles (Encoder side)
    inputs = [doc for doc in examples["article"]]
    model_inputs = tokenizer(
        inputs, 
        max_length=1024, 
        truncation=True, 
        padding="max_length"
    )
    
    # 2. Tokenize the target summaries (Decoder side)
    # The 'text_target' parameter replaces the old 'as_target_tokenizer' logic
    labels = tokenizer(
        text_target=examples["highlights"], 
        max_length=128, 
        truncation=True, 
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [10]:
tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(preprocess_function, batched=True, remove_columns=val_data.column_names)

Map:   0%|          | 0/59000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [11]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [12]:
# 6. Updated Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-summarizer-100k",
    eval_strategy="steps",        # Changed from evaluation_strategy
    eval_steps=1000,
    learning_rate=2e-4, 
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, 
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=7, 
    predict_with_generate=True,
    fp16=True, 
    logging_steps=100,
    report_to="none" 
)

In [13]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    # Remove tokenizer=tokenizer
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
1000,30.453308,7.295876
2000,30.091150,7.249899
3000,29.825862,7.240412
4000,29.889824,7.222963
5000,29.748054,7.212775


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

In [ ]:
model.save_pretrained("./best-summarizer-weights")

In [ ]:
import shutil
import os

# 1. تحديد الفولدر اللي هنحفظ فيه
model_dir = "/kaggle/working/summarizer_model"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# 2. حفظ الموديل والـ Tokenizer
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

# 3. ضغط الفولدر لملف Zip واحد
zip_name = "/kaggle/working/my_lora_model"
shutil.make_archive(zip_name, 'zip', model_dir)

print(f"✅ جاهز للتحميل: {zip_name}.zip")